# CAR — Compressibility-Aware Routing

Train a model that routes tokens by *prediction error*. Easy tokens go cheap. Hard tokens get the full path.

**Run all cells in order. Cell 1 first, then 2, then 3, etc.**

In [ ]:
# Cell 1: Install dependencies (run once, wait for it to finish)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "wandb"])
print("✅ wandb installed")

In [ ]:
# Cell 2: Imports
import os, sys, time, math, traceback
import torch
import torch.nn as nn
from torch.optim import AdamW
import wandb

print(f"torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3: Config — change SCALE here
SCALE = "small"  # tiny=128d/4l, small=256d/4l, medium=512d/8l

SCALES = {
    "tiny":   dict(d_model=128, n_layers=4, d_state=32),
    "small":  dict(d_model=256, n_layers=4, d_state=32),
    "medium": dict(d_model=512, n_layers=8, d_state=64),
}

cfg = SCALES[SCALE]
VOCAB_SIZE = 4096
SEQ_LEN    = 128      # short = fast
BATCH_SIZE = 8
GRAD_ACCUM = 4
MAX_STEPS  = 5000
EVAL_EVERY = 500
SAVE_EVERY = 1000
LR         = 3e-4
THRESHOLD  = 1.0      # higher = more tokens go cheap path

print(f"Scale: {SCALE} | d_model={cfg['d_model']} | n_layers={cfg['n_layers']}")
print(f"Batch={BATCH_SIZE}×{GRAD_ACCUM} | SEQ={SEQ_LEN} | steps={MAX_STEPS}")

In [ ]:
# Cell 4: Model classes — no external dependencies, pure torch

class RMSNorm(nn.Module):
    """Works on any PyTorch version."""
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d))
        self.eps = eps
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * norm * self.scale


class SSMLayer(nn.Module):
    """Feedforward layer that acts like an SSM."""
    def __init__(self, d_model, d_state=64, expand=2):
        super().__init__()
        d_inner = d_model * expand
        self.proj = nn.Sequential(
            nn.Linear(d_model, d_inner),
            nn.Tanh(),
            nn.Linear(d_inner, d_state),
        )
        self.out = nn.Linear(d_state, d_model)
        self.h = None  # persistent state
    def forward(self, x):
        B, T, D = x.shape
        state = self.proj(x)           # (B,T,d_state)
        h_new = state.mean(dim=1)       # (B,d_state)
        if self.h is not None:
            h_new = 0.9 * self.h + 0.1 * h_new  # EMA update
        self.h = h_new.detach()
        out = self.out(h_new).unsqueeze(1).expand(-1, T, -1)  # (B,T,D)
        return out


class CheapPath(nn.Module):
    """Cheap MLP — used for easy tokens."""
    def __init__(self, d_model, ratio=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model // ratio),
            nn.GELU(),
            nn.Linear(d_model // ratio, d_model),
        )
    def forward(self, x):
        return self.net(x)


class CARState(nn.Module):
    """Persistent state that predicts next token. Low error = cheap routing."""
    def __init__(self, d_model, d_state=64):
        super().__init__()
        self.update = nn.GRUCell(d_model, d_state)
        self.predict = nn.Sequential(
            nn.Linear(d_state, d_state * 2),
            nn.Tanh(),
            nn.Linear(d_state * 2, d_model),
        )
        self.d_state = d_state
    def init_state(self, B, device, dtype):
        return torch.zeros(B, self.d_state, device=device, dtype=dtype)
    def forward(self, x, h):
        summary = x.mean(dim=1)           # (B,D)
        h_new = self.update(summary, h)  # (B,d_state)
        pred = self.predict(h_new)        # (B,D) — predicts next token
        return h_new, pred


class CARModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=4, d_state=64,
                 cheap_ratio=4, threshold=1.0):
        super().__init__()
        self.threshold = threshold
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "ssm":   SSMLayer(d_model, d_state),
                "cheap": CheapPath(d_model, cheap_ratio),
                "norm":  RMSNorm(d_model),
            })
            for _ in range(n_layers)
        ])
        self.car_state = CARState(d_model, d_state)
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight  # weight tying

    def forward(self, input_ids, targets=None):
        B, T = input_ids.shape
        x = self.embed(input_ids)   # (B,T,D)
        h = self.car_state.init_state(B, x.device, x.dtype)
        cheap_ratios = []
        for layer in self.layers:
            xn = layer["norm"](x)
            h, pred = self.car_state(xn, h)
            # Prediction error: how badly did the state predict this token?
            with torch.no_grad():
                pred_expanded = pred.unsqueeze(1).expand(-1, T, -1)  # (B,T,D)
                error = (pred_expanded - xn).pow(2).mean(-1).mean(-1)  # (B,)
            cheap_w = torch.sigmoid(self.threshold - error)  # (B,)
            cheap_ratios.append(cheap_w.mean().item())
            ssm_out   = layer["ssm"](xn)
            cheap_out = layer["cheap"](xn)
            cw = cheap_w.view(B, 1, 1)  # (B,1,1) broadcasts over T
            x = x + cw * cheap_out + (1 - cw) * ssm_out
        x = self.norm_f(x)
        logits = self.lm_head(x)
        result = {"logits": logits,
                  "cheap_ratio": sum(cheap_ratios)/max(len(cheap_ratios),1)}
        if targets is not None:
            result["loss"] = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1)
        return result
    def param_count(self):
        return sum(p.numel() for p in self.parameters())

print("✅ Model classes OK")

In [ ]:
# Cell 5: Build model and test one forward pass
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = CARModel(
    vocab_size=VOCAB_SIZE,
    d_model=cfg["d_model"],
    n_layers=cfg["n_layers"],
    d_state=cfg["d_state"],
    cheap_ratio=4,
    threshold=THRESHOLD,
).to(device)

n_params = model.param_count() / 1e6
print(f"Model: {n_params:.2f}M params")

# Test forward pass
print("Testing forward pass...", end="", flush=True)
model.eval()
with torch.no_grad():
    test_ids = torch.randint(0, VOCAB_SIZE, (2, SEQ_LEN), device=device)
    test_targets = torch.randint(0, VOCAB_SIZE, (2, SEQ_LEN), device=device)
    out = model(test_ids, test_targets)
    print(f" done\n  logits: {out['logits'].shape} | loss: {out['loss'].item():.4f} | cheap: {out['cheap_ratio']:.2%}")
model.train()
print("✅ Forward pass OK — model runs!")

In [ ]:
# Cell 6: Synthetic data — no downloads needed
torch.manual_seed(42)
N_TRAIN, N_VAL = 50000, 5000
train_data = [torch.randint(0, VOCAB_SIZE, (SEQ_LEN,)) for _ in range(N_TRAIN)]
val_data   = [torch.randint(0, VOCAB_SIZE, (SEQ_LEN,)) for _ in range(N_VAL)]
print(f"Train: {len(train_data)} | Val: {len(val_data)} | ✅ No downloads!")

In [ ]:
# Cell 7: W&B login — get your key at https://wandb.ai/authorize
wandb.login()

In [ ]:
# Cell 8: Train!
run = wandb.init(
    project="neu",
    name=f"car-{SCALE}-{time.strftime('%m%d-%H%M')}",
    config=dict(scale=SCALE, **cfg, vocab=VOCAB_SIZE, seq=SEQ_LEN,
                batch=BATCH_SIZE, accum=GRAD_ACCUM, steps=MAX_STEPS,
                lr=LR, threshold=THRESHOLD, device=str(device)),
    mode="online",
)
wandb.define_metric("step")
wandb.define_metric("val_loss", step_metric="eval_step")

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.1, betas=(0.9, 0.95))

def sched(step):
    w = 100
    if step < w: return max(0.1, step/w)
    p = (step-w)/max(MAX_STEPS-w,1)
    return max(1e-5/LR, 0.5*(1+math.cos(math.pi*p)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, sched)

print(f"""
============================================================
  CAR Training | {SCALE} | {n_params:.2f}M params | {device}
  Batch: {BATCH_SIZE}×{GRAD_ACCUM} | Steps: {MAX_STEPS:,} | Cheap threshold: {THRESHOLD}
============================================================
""")

step, accum, t0, total_tokens = 0, 0, time.time(), 0
model.train()

print(f"{'step':>6} | {'loss':>8} | {'cheap':>6} | {'lr':>9} | {'tok/s':>8}")
print("-"*65)

try:
    while step < MAX_STEPS:
        for seq in train_data:
            if step >= MAX_STEPS: break
            ids = seq.unsqueeze(0).to(device)
            tgt = seq.unsqueeze(0).to(device)

            r = model(ids, tgt)
            loss = r["loss"] / GRAD_ACCUM
            loss.backward()
            accum += 1
            cheap = r["cheap_ratio"]

            if accum >= GRAD_ACCUM:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                accum = 0

                total_tokens += ids.numel() * GRAD_ACCUM
                dt = time.time()-t0
                tok_s = total_tokens / max(dt, 0.1)
                lr_now = scheduler.get_last_lr()[0]

                if step % 10 == 0:
                    print(f"{step:>6} | {loss.item()*GRAD_ACCUM:>8.4f} | "
                          f"{cheap:>6.2%} | {lr_now:>9.2e} | {tok_s:>8.0f}")

                wandb.log({"step": step, "loss": loss.item()*GRAD_ACCUM,
                           "cheap_ratio": cheap, "lr": lr_now,
                           "tokens_per_sec": tok_s}, step=step)

                if step > 0 and step % EVAL_EVERY == 0:
                    model.eval()
                    vl = []
                    with torch.no_grad():
                        for s in val_data[:25]:
                            iv = s.unsqueeze(0).to(device)
                            tv = s.unsqueeze(0).to(device)
                            vl.append(model(iv, tv)["loss"].item())
                    vloss = sum(vl)/len(vl)
                    wandb.log({"eval_step": step, "val_loss": vloss}, step=step)
                    print(f"  [EVAL @{step:>6}] val={vloss:.4f}")
                    model.train()

                if step > 0 and step % SAVE_EVERY == 0:
                    ckpt = f"car-{SCALE}-s{step}.pt"
                    torch.save({"step": step, "state": model.state_dict()}, ckpt)
                    wandb.save(ckpt)
                    print(f"  [SAVE @{step}] → {ckpt}")

                step += 1

    torch.save({"step": step, "state": model.state_dict()}, f"car-{SCALE}-final.pt")
    wandb.finish()
    print(f"\n✅ Done! {total_tokens/1e6:.1f}M tokens in {time.time()-t0:.0f}s")

except Exception as e:
    print(f"\n❌ CRASH @{step}: {e}")
    traceback.print_exc()
    try:
        torch.save({"step": step, "state": model.state_dict()}, f"car-{SCALE}-crash.pt")
        print("Crash checkpoint saved")
    except: pass
    wandb.finish()
    raise

## What to watch

| Metric | Target |
|--------|--------|
| cheap_ratio | 50–80% — most tokens should go cheap |
| loss | Decreasing steadily |
| val_loss | Tracks training loss |

**cheap_ratio < 30%:** Lower threshold (e.g. 0.5) in Cell 3 and re-run Cell 8.
**cheap_ratio > 95%:** Raise threshold (e.g. 2.0) in Cell 3 and re-run Cell 8.